# Data Preparation
## Getting Started
This tutorial focuses on data, including a brief discussion on how to best prepare your data so it works well with the `chainladder` package. 

Be sure to make sure your packages are updated. For more info on how to update your pakages, visit [Keeping Packages Updated](https://chainladder-python.readthedocs.io/en/latest/library/install.html#keeping-packages-updated).

In [0]:
# Black linter, optional
# import jupyter_black as jb

# jb.load()

import pandas as pd
import numpy as np
import chainladder as cl
import matplotlib.pyplot as plt

print("pandas: " + pd.__version__)
print("numpy: " + np.__version__)
print("chainladder: " + cl.__version__)

## Disclaimer
Note that a lot of the examples shown might not be applicable in a real world scenario, and is only meant to demonstrate some of the functionalities included in the package. The user should always follow all applicable laws, the Code of Professional Conduct, applicable Actuarial Standards of Practice, and exercise their best actuarial judgement.

## Converting Triangle Data into Long Format

One of the most commonly asked questions is that if the data needs to be in the tabular long format as opposed to the already processed triangle format when we are loading the data for use. 

Unfortunately, the chainladder package requires the data to be in long form. 

Suppose you have a wide triangle.

In [0]:
df = cl.load_sample("raa").to_frame(origin_as_datetime=True)
df

You can use pandas to unstack the data into the wide long format.

In [0]:
df = df.unstack().dropna().reset_index()
df.head(10)

Let's clean up our column names before we get too far.

In [0]:
df.columns = ["age", "origin", "values"]
df.head()

Next, we will need a valuation column (think Schedule P style triangle).

In [0]:
df["valuation"] = (df["origin"].dt.year + df["age"] / 12 - 1).astype(int)
df.head()

Now, we are finally ready to load it into the chainladder package!

In [0]:
cl.Triangle(
    df, origin="origin", development="valuation", columns="values", cumulative=True
)

## Sparse Triangles
By default, the chainladder `Triangle` is a wrapper around a numpy array.  Numpy is optimized for high performance and this allows chainladder to achieve decent computational speeds. Despite being fast, numpy can become memory inefficient with triangle data because triangles are inherently sparse (when memory is being allocated yet no data is stored).

The lower half of a an incomplete triangle is generally blank and that means about 50% of an array size is wasted on empty space. As we include granular index and column values in our `Triangle`, the sparsity of the triangle increases further consuming RAM unnecessarily. Chainladder automatically eliminates this extraneous consumption of memory by resorting to a sparse array representation when the Triangle becomes sufficiently large.

Let's load the `prism` dataset and include each claim number in the index of the Triangle. The dataset is claim level and includes over 130,000 triangles.

In [0]:
prism = cl.load_sample("prism")
prism

Let's also look at the array representation of the Triangle and notice how it is no longer a numpy array, but instead a sparse array.

In [0]:
prism.values

The sparse array consumes about 4.6Mb of memory. We can also see its density is very low, this is because individual claims will at most exist in only one origin period.  Let's approximate the size of this Triangle assuming we used a dense array representation.  Approximation can be done by assuming 8 bytes (for float64) of memory are used for each cell in the array.

In [0]:
print("Dense array size:", np.prod(prism.shape) / 1e6 * 8, "MB.")
print("Sparse array size:", prism.values.nbytes / 1e6, "MB.")
print(
    "Dense array is",
    round((np.prod(prism.shape) / 1e6 * 8) / (prism.values.nbytes / 1e6), 1),
    "times larger!",
)

## Incremental vs Cumulative Triangles
Cumulative triangles are naturally denser than those stored in an incremental fashion. While almost all actuarial techniques rely on cumulative triangles, it may be worthwhile to maintain and manipulate triangles as incremental triangles until you are ready to apply a model.

In [0]:
prism.values

In [0]:
prism = prism.incr_to_cum()
prism.values

Our incremental triangle is under 5MB, but when we convert to a cumulative triangle it becomes an astonishingly large 219MB and this is despite still maintaining a sparsity of under 0.3%!

## Claim-Level Data
The sparse representation of triangles allows for substantially more data to be pushed through chainladder. This gives us some nice capabilities that we would not otherwise be able to do with aggregate data.

For example, we can now drill into the individual claim makeup of any cell in our Triangle.  Let's look at January 2017 claim details at age 12.

In [0]:
claims = prism[prism.origin == "2017-01"][prism.development == 12].to_frame(
    origin_as_datetime=True
)
claims[abs(claims).sum(axis="columns") != 0].reset_index()

We can also examine the data as the usual aggregated Triangle, by applying `sum()`. We'll also apply `grain()` so we can better visualize the data.

In [0]:
plt.plot(prism["Paid"].sum().grain("OYDM").to_frame(origin_as_datetime=True).T)

With claim level data, we can set a claim large loss cap or create an excess Triangle on the fly.

In [0]:
prism["Capped 100k Paid"] = cl.minimum(prism["Paid"], 100000)
prism["Excess 100k Paid"] = prism["Paid"] - prism["Capped 100k Paid"]

In [0]:
plt.plot(
    prism["Excess 100k Paid"].sum().grain("OYDM").to_frame(origin_as_datetime=True).T
)

### Claim-Level IBNR Estimates

Let's see how we can use the API to create claim-level IBNR estimates.  When using aggregate actuarial techniques, it really makes sense to perform the model fitting at an aggregate level.  

We use aggregate data to fit the model to generate reasonable development patterns.

In [0]:
agg_data = prism.sum()[["Paid", "reportedCount"]]
model_cl = cl.Chainladder().fit(agg_data)

With the fitted model, we are not limited to predicting ultimates at the aggregated grain. Let's predict chainladder ultimates at a claim level. Here, we are using `model_cl`, which was built using `agg_data` to make prediction on `prism`, which is claim-level data.

In [0]:
cl_ults = model_cl.predict(prism[["Paid", "reportedCount"]]).ultimate_
cl_ults

We could stop here, but let's try a Bornhuetter-Ferguson method as well. We will infer an a-priori severity from our chainladder model, `model_cl` above.

In [0]:
plt.plot(
    (model_cl.ultimate_["Paid"] / model_cl.ultimate_["reportedCount"]).to_frame(
        origin_as_datetime=True
    ),
)

40K seems a reasonable a-priori (at least for the last two years or so, between 2016 - 2017).

Now, let's fit an aggregate Bornhuetter-Ferguson model.  Like the chainladder example, we fit the model in aggregate (summing all claims) to create a stable model from which we can generate granular predictions.  We will use our Chainladder ultimate claim counts as our `sample_weight` (exposure) for the BornhuetterFerguson method.

In [0]:
paid_bf = cl.BornhuetterFerguson(apriori=40000).fit(
    X=prism["Paid"].sum().incr_to_cum(), sample_weight=cl_ults["reportedCount"].sum()
)

In [0]:
plt.bar(
    paid_bf.ultimate_.grain("OYDM").to_frame(origin_as_datetime=False).index.year,
    paid_bf.ultimate_.grain("OYDM").to_frame(origin_as_datetime=False)["2261-12"],
)

We can now create claim-level BornhuetterFerguson predictions using our claim-level Triangle.  Ideally, the results should tie to the aggregate results.

In [0]:
bf_ults = paid_bf.predict(
    prism["Paid"].incr_to_cum(), sample_weight=cl_ults["reportedCount"]
).ultimate_

In [0]:
plt.bar(
    bf_ults.sum().grain("OYDM").to_frame(origin_as_datetime=False).index.year,
    bf_ults.sum().grain("OYDM").to_frame(origin_as_datetime=False)["2261-12"],
)

In [0]:
import doctest
import sys

def test_func():
    """
    This is a test:
    >>> 2 + 2
    4    
    """
    return None

result = doctest.testmod(verbose=True)
if result.failed > 0:
    sys.exit(1)